# Lecture 02 Exercise: Learn an Answer Format with LoRA and Compare Before and After Training

Using a fictional course FAQ, briefly train a small instruction-tuned model with LoRA and compare **what changes and what stays the same** before and after training. Evaluation separates **format compliance rate** (whether it follows the JSON format with `answer` and `needs_review`) and **content accuracy**.

**What to Check in This Exercise**

1. A small model may fail to follow format instructions even when they are in the prompt (section 3, before training).
2. In the original Japanese exercise, it followed the format almost consistently after a few minutes of LoRA training; check whether your run does too (section 5, after training).
3. A correct format does not guarantee sound judgment when asked about things absent from the FAQ (whether to request confirmation or make an assertion; sections 5 and 6).
4. Advanced: in the original Japanese exercise, loading the same model with 4-bit quantization changed outputs even without training; compare your own results (section 8).

**Workflow**

| Section | Contents | Estimated Time (T4, Original Japanese Exercise) |
| --- | --- | --- |
| 0–3 | Environment, settings, data generation, model loading, before-training evaluation | About 5 minutes |
| 4–7 | Training, after-training evaluation, knowledge questions, comparison table and submission | About 10 minutes (including 4–5 minutes of training) |
| 8 (optional) | Load in 4-bit and compare | About 2 minutes |

Running 0–3 before class to check the environment will help avoid delays during class (optional). Colab loses variables when the session disconnects; if that happens, start again from 0 on the day.

**Requirements**: Google Colab (T4 GPU, free tier). Model: `Qwen/Qwen3.5-0.8B` (Apache-2.0). This notebook generates fictional data, with no real deadlines or URLs.

**How to Read**: Each section explains "What to Do", "What to Look For", and "Troubleshooting". Run cells in order from the top. Skipping cells causes missing-variable errors in later cells.

**Translation note**: The timing, loss, and output observations quoted here come from the original Japanese exercise. Results may differ with the translated FAQ and questions; these observations have not been remeasured for this translation.

Version: 1.1 (2026-10-18)

## 0. Check the Environment and Install Libraries

**What to Do**: Check that a GPU is allocated and install libraries pinned to verified versions.

**What to Look For**

- The first cell displays `Tesla T4`.
- The second cell ends with a list of versions (such as `transformers: 5.18.0`). Installation takes about 1 minute.

**Troubleshooting**

- "No GPU found": select **Runtime → Change runtime type → T4 GPU** and run the cell again. If demand prevents allocation, try again later.
- Red pip `ERROR: pip's dependency resolver ...` messages (about `cudf` or `pyarrow`) warn of conflicts with other packages preinstalled in Colab. They do not affect this exercise, so continue.

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU runtimes do not have nvidia-smi
    gpu_info = ""
print(gpu_info or "No GPU found. Select Runtime → Change runtime type → T4 GPU.")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Versions verified on a Colab T4 (Python 3.13, torch 2.11.0+cu130). Use the torch bundled with Colab
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 requires transformers 5.x
    "peft": "0.21.1",           # LoRA implementation
    "trl": "1.14.2",            # Training loop (SFTTrainer)
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # The bundled torchao (0.10) is incompatible with peft 0.21 and causes ImportError during training. Remove it since it is unused
%pip install -q {spec}

# Dependency warnings about cudf-cu13 and pyarrow appear as ERROR, but can be ignored because cudf is unused
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. Settings

**What to Do**: Collect the model name, number of training epochs (passes through the data), and other settings in a single dictionary, `CONFIG`. This dictionary is recorded as is in the submission file in section 7 so you can later identify "the settings used for each experiment".

- Fixing `revision` ensures the same weights are used whenever you run the notebook. This keeps conditions consistent for before/after and cross-person comparisons.
- `num_train_epochs: 20` means 20 passes through the training data (14 examples). With 2 steps per pass, this totals 40 steps. The original Japanese run took 4–5 minutes on a T4; timing may differ with the translated data.
- `gen: {"do_sample": False}` uses greedy generation. This fixes generation conditions before and after training and removes the effect of sampling variation.

Setting `USE_DRIVE` to `True` mounts Google Drive and saves the trained adapter and result files to Drive as well. Use this to retain results if the session disconnects. Leave it `False` if you will finish in one session.

**What to Look For**: The settings are displayed as JSON.

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # Small instruction-tuned model (Apache-2.0). Fits on a T4 without quantization
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # Pinned to use the same weights on every run
    "seed": 42,
    "num_train_epochs": 10,  # Number of passes over training data (14 examples). 2 steps per pass gives 40 steps (4–5 minutes on a T4 in the original Japanese run)
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA matrix dimension (smaller means fewer trainable parameters)
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # Fix evaluation to greedy generation (consistent before/after conditions)
    "work_dir": "/content/qlora_lab",                          # Result file location (visible in the file list on the left)
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # Save location when USE_DRIVE=True
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # Set True to mount Google Drive and save adapters and outputs
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 10,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [4]:

from google.colab import files
uploaded = files.upload()


Saving glossary_sft.jsonl to glossary_sft.jsonl


In [5]:

import json

with open("glossary_sft.jsonl", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

    print("Examples:", len(data))
    print("First record keys:", list(data[0].keys()))


Examples: 116
First record keys: ['messages']


## 2. Generate a Fictional FAQ and Training Data

**What to Do**: Create training (train), validation (valid), and evaluation (test) data from a fictional FAQ with 5 items.

Model input is "FAQ + question"; the desired output is one JSON line: `{"answer": <string>, "needs_review": <boolean>}`. In the example filename `student_id_assignment_number.pdf`, replace `student_id` with your student ID and `assignment_number` with the assignment number.

- For questions with evidence in the FAQ, answer with the FAQ text and set `needs_review: false`.
- For questions absent from the FAQ, answer "This FAQ does not contain that information. Confirmation from the course organizers is needed." and set `needs_review: true`.

**Data Splits**

- Paraphrases asking about the same FAQ item are assigned to train / valid / test **by template**, so text seen during training does not appear unchanged in test.
- test has 21 questions (15 with evidence, 6 without). Additional test-only questions ensure this count.
- Expected answers (the `needs_review` value and required answer keywords) are held separately in `gold` and are not passed to the model.
- train has 14 examples (10 with evidence, 4 without), and valid has 7. With only 5 FAQ items, training data is limited, so the same examples are trained on repeatedly. The number of repetitions is controlled by the number of epochs (`num_train_epochs`) in section 4, rather than duplicating data. The standard approach is to put only actual examples in data files and control training volume through training settings.

**What to Look For**: `train=14 valid=7 test=21 gold=21` and one training example (question and desired output) are displayed. Duplicate questions anywhere across train / valid / test cause an error here.

In [6]:

import json
from datasets import Dataset

# Load glossary dataset
with open("glossary_sft.jsonl", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

    # Split into training and validation
    dataset = Dataset.from_list(data)
    dataset = dataset.shuffle(seed=42)

    train_ds = dataset.select(range(104))
    valid_ds = dataset.select(range(104, 116))

    print("Train:", len(train_ds))
    print("Valid:", len(valid_ds))
    print("First example:", train_ds[0]["messages"])


Train: 104
Valid: 12
First example: [{'role': 'user', 'content': 'Briefly explain Evaluation and improvement cycle.'}, {'role': 'assistant', 'content': 'A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve prompts, data, models, and tools based on the results. In this course, we repeat this cycle across lectures using a learning support agent.'}]


In [8]:
# Show 8 glossary examples
for i in range(8):
    messages = train_ds[i]["messages"]
    question = messages[0]["content"]
    answer = messages[1]["content"]
    print(f"{i+1}. Question: {question}")
    print(f"Answer: {answer[:200]}")
    print("---")

1. Question: Briefly explain Evaluation and improvement cycle.
Answer: A development approach that repeats build → measure → improve. Since AI application outputs vary probabilistically, judge quality using evaluation data and metrics rather than intuition, then improve 
---
2. Question: What is devcontainer?
Answer: A mechanism for defining a development environment as a container. Specify the Python version, extensions, and startup commands in .devcontainer/devcontainer.json so everyone opening Codespaces gets t
---
3. Question: Briefly explain fork.
Answer: Copying someone else's entire repository to your own account. In this course, fork the course repository and work and submit within your own fork.
---
4. Question: What is AI engineering?
Answer: The technical field of designing, building, evaluating, and operating useful applications and systems using pretrained foundation models (mainly LLMs) as components. It focuses on selecting models, co
---
5. Question: What is LoRA?
Answe

In [10]:
# Reference answers from the glossary
references = {
    row["messages"][0]["content"]: row["messages"][1]["content"]
    for row in data
}

test_cases = [
    # Terms in glossary
    {
        "type": "known",
        "question": "Define LoRA.",
        "expected": references["What is LoRA?"]
    },
    {
        "type": "known",
        "question": "What does AI engineering mean?",
        "expected": references["What is AI engineering?"]
    },
    # Reworded questions
    {
        "type": "reworded",
        "question": "How does LoRA reduce the number of trainable parameters?",
        "expected": references["What is LoRA?"]
    },
    {
        "type": "reworded",
        "question": "Why do developers use a devcontainer?",
        "expected": references["What is devcontainer?"]
    },
    {
        "type": "reworded",
        "question": "Explain the build-measure-improve cycle.",
        "expected": references["Briefly explain Evaluation and improvement cycle."]
    },
]

# Select terms with no matching glossary entry
glossary_questions = [
    row["messages"][0]["content"].lower()
    for row in data
]

candidates = [
    "Speculative decoding",
    "Mixture of Depths",
    "FlashAttention-3",
    "Diffusion Policy"
]

unknown_terms = [
    term for term in candidates
    if not any(term.lower() in q for q in glossary_questions)
][:2]

assert len(unknown_terms) == 2

for term in unknown_terms:
    test_cases.append({
        "type": "unknown",
        "question": f"What is {term}?",
        "expected": "Not covered in the glossary"
    })

print("Evaluation questions:", len(test_cases))

for case in test_cases:
    print(case["type"], "-", case["question"])

Evaluation questions: 7
known - Define LoRA.
known - What does AI engineering mean?
reworded - How does LoRA reduce the number of trainable parameters?
reworded - Why do developers use a devcontainer?
reworded - Explain the build-measure-improve cycle.
unknown - What is Speculative decoding?
unknown - What is Mixture of Depths?


## 3. Load the Model and Evaluate Before Training

**What to Do**: Load the base model without quantization (32-bit), have it answer the 21 test questions, and score the results.

The prompt (`SYSTEM`) says "Always return `answer` and `needs_review` in one JSON line". The **format has already been instructed**. Before training, check whether the model follows this instruction.

**Scoring** (the same functions are used in section 5)

- **Format**: The output parses as JSON, has exactly two keys (`answer` and `needs_review`), and uses string and boolean types respectively.
- **Content**: `needs_review` matches the expected answer, and `answer` contains the required keywords (for example, "September 30" and "23:59").

Content scoring requires a matching `needs_review` value. Outputs with a missing key therefore receive `Fail` for content even if the wording is correct. Check the wording yourself in the table. The format-failure codes mean: `json_error` = JSON parsing failed; `not_object` = the JSON value is not an object; `extra_keys` = unexpected keys are present; `missing_key` = a required key is absent; `wrong_type` = an answer or review flag has the wrong type.

`Qwen/Qwen3.5-0.8B` also supports images, but loading it with `AutoModelForCausalLM` uses only the text component.

**What to Look For**

- First cell: a message such as `Weight memory (GB): 3.01`. Downloading the model (about 1.75GB) takes about 1 minute.
- Second cell: a one-line summary (`format_rate`, `content_rate`, etc.) and a table of outputs for 21 questions. Evaluation takes about 1 minute. In the original Japanese run, many before-training outputs were missing `needs_review`; check which keys are missing in your run.

**Harmless Warnings**

- `You are sending unauthenticated requests to the HF Hub`: this means the model is being downloaded without signing in to Hugging Face. Public models can be downloaded without signing in. Signing in is needed in the following cases, regardless of model size:
  - Using a model that requires agreement to terms (a gated model, such as Llama or Gemma) or a private repository.
  - Uploading your own models or data to the Hub.
  - Sending many requests in a short time and hitting a limit. Without signing in, the limit is 3,000 file retrievals per IP address per 5 minutes (published figure as of September 2025; [Hub rate limits](https://huggingface.co/docs/hub/rate-limits)). Signing in raises it.

  This notebook's model is public and requires no agreement. Loading it makes at most a few dozen requests, well below the limit. If `429 Too Many Requests` does occur, the library reads the waiting time and retries automatically. If signing in becomes necessary, register a Hugging Face token named `HF_TOKEN` using the key icon (Secrets) on Colab's left sidebar and allow this notebook to access it; it will be used automatically.
- `... is falling back to its reference PyTorch implementation ...`: a fast implementation for some of this model's layers (linear attention) is not installed, so a slower but correct implementation is used. About 4 lines appear.

**Read Before Continuing**: Check the table for "which keys are missing" and "whether the wording agrees with the FAQ". In section 5, compare this table alongside outputs after training. Original outputs are also saved in `/content/qlora_lab/eval_before.json` (open it from the file list on the left).

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(
    CONFIG["base_model"],
    revision=CONFIG["revision"]
)

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    revision=CONFIG["revision"],
    dtype=torch.float32
).to(DEVICE)

model.config.use_cache = True

def generate(model, question):
    messages = [{"role": "user", "content": question}]
    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            pad_token_id=tok.pad_token_id,
            **CONFIG["gen"]
        )
    return tok.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

print("Model loaded:", CONFIG["base_model"])
print("Device:", DEVICE)
print("BEFORE Training:")
print(generate(model, test_cases[0]["question"]))

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Model loaded: Qwen/Qwen3.5-0.8B
Device: cuda
BEFORE Training:


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


**LoRA** stands for **Low-Rank Adaptation**. It is a technique used in **Neural Networks** to fine-tune large pre-trained models (like Large Language Models, LLMs) with significantly less computational cost and training time.

### Core Concept
Instead of training a new model from scratch, LoRA allows you to modify a small, fixed portion of the weights of a large model. This small modification is called a **LoRA adapter**.


In [14]:
import pandas as pd
import json

def evaluate_glossary(current_model, label):
    results = []
    for case in test_cases:
        answer = generate(current_model, case["question"])
        results.append({
            "type": case["type"],
            "question": case["question"],
            "expected": case["expected"],
            "output": answer
        })

    path = WORK / f"glossary_eval_{label}.json"
    path.write_text(
        json.dumps(results, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"{label}: {len(results)} questions evaluated")
    display(pd.DataFrame(results))
    return results

before_rows = evaluate_glossary(model, "before")

before: 7 questions evaluated


,type,question,expected,output
0,known,Define LoRA.,"A PEFT method that freezes pretrained weights,...",**LoRA** stands for **Low-Rank Adaptation**. I...
1,known,What does AI engineering mean?,"The technical field of designing, building, ev...",**AI Engineering** (Artificial Intelligence En...
2,reworded,How does LoRA reduce the number of trainable p...,"A PEFT method that freezes pretrained weights,...",LoRA (Low-Rank Adaptation) reduces the number ...
3,reworded,Why do developers use a devcontainer?,A mechanism for defining a development environ...,Developers use **DevContainers** (often called...
4,reworded,Explain the build-measure-improve cycle.,A development approach that repeats build → me...,The **Build-Measure-Improve (BMI)** cycle is a...
5,unknown,What is Speculative decoding?,Not covered in the glossary,**Speculative decoding** is a technique used i...
6,unknown,What is Mixture of Depths?,Not covered in the glossary,**Mixture of Depths** (often referred to as **...


## 4. LoRA Training (Original Japanese Run: 4–5 Minutes on a T4)

### LoRA versus QLoRA

- **LoRA**: Freeze the base-model weights and train only small matrices (adapters) added to some layers.
- **QLoRA**: Keep base-model weights quantized to 4-bit and train only adapters in the same way. The adapters themselves are not quantized.

QLoRA reduces the memory used to hold base-model weights. It uses about a quarter of the memory of 16-bit, so models too large to fit on a GPU otherwise can be trained (for 7 billion parameters, about 14GB at 16-bit versus about 3.5GB at 4-bit).

| Aspect | LoRA (no quantization) | QLoRA (4-bit) |
| --- | --- | --- |
| Weight memory | Large | Small (about 1/4 of 16-bit) |
| Base-model outputs | Unchanged | May change due to quantization error; the size of the effect depends on the model |
| Training speed | Baseline | Not faster. Often slower because 4-bit weights must be converted back for each computation |
| Dependencies | peft | peft plus bitsandbytes (requires a CUDA GPU) |

Notes for implementing QLoRA:

- Pass `BitsAndBytesConfig(load_in_4bit=True, ...)` when loading, then run `prepare_model_for_kbit_training(model)` before adding LoRA.
- Storage precision (4-bit) and compute precision (`bnb_4bit_compute_dtype`, such as 16-bit) are separate settings.
- Compare before and after training using the same quantization conditions for the base model. Measuring without quantization before training and with 4-bit afterward mixes quantization and training effects.
- Adapters are created for the base model used in training. Record the base-model ID, revision, and quantization settings together when distributing or reusing them.
- 4-bit training is discouraged for some models. One [training-library guide](https://unsloth.ai/docs/models/qwen3.5/fine-tune) discourages 4-bit training of Qwen3.5 because its quantization error is larger than usual.

This exercise's model fits on a T4 without quantization, so we train with LoRA. Consider LoRA first if the model fits; if it does not, consider QLoRA or a smaller model. You can examine memory and output differences when loading in 4-bit in section 8 (Advanced).

### Run Training

**What to Do**: Convert the 14 train examples to chat format (system / user / assistant), attach LoRA, and train for 20 epochs (40 steps). Save the trained adapter in `adapter/`.

- `target_modules` specifies layers where LoRA is inserted. We target attention projections (such as `q_proj`) and MLP projections (such as `gate_proj`). This model uses linear attention in 18 of its 24 layers; only the remaining 6 have `q_proj` and similar projections. All layers have MLPs.
- Batch 4 × gradient accumulation 2 = 8 examples per step. With 14 train examples, each epoch has 2 steps (8 and 6 examples), giving 40 steps over 20 epochs. The model sees each training example exactly 20 times. Learning-rate warmup and cosine decay apply over these 40 steps.
- The trained model is stored in `tuned`. Since LoRA is inserted into the base-model layers, generation through the original variable `model` also includes adapters. Temporarily disable adapters as in section 6 when you need before-training outputs.

**What to Look For**

- `trainable params: ... || all params: ... || trainable%: ...`: a feature of LoRA is that only a few percent of all parameters are trained.
- The `Training Loss` table (every 10 steps): in the original Japanese run, loss started around 1.5 and dropped to 0.01–0.02 by step 40. The translated data may give different values. If training ends without a decrease, check whether you changed the settings.
- `Training time: ... seconds | Steps: 40 | Epochs: 20.0`: the original Japanese run took 240–290 seconds on a T4; timing may differ with the translated data. Steps and epochs confirm that 14 training examples × 20 passes were run as calculated above.

**Harmless Warning**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` means end-token settings were aligned with the tokenizer.

**Troubleshooting**: If training stops midway or cannot run, read 4b below.

In [16]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
import time

model.config.use_cache = False

lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

sft = SFTConfig(
    output_dir=str(WORK / "glossary_ckpt"),
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    fp16=False,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"]
)

trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora,
    processing_class=tok
)

trainer.model.print_trainable_parameters()

t0 = time.time()
trainer.train()

training_seconds = time.time() - t0

print("Training time:", round(training_seconds, 2))
print("Steps:", trainer.state.global_step)
print("Epochs:", trainer.state.epoch)

tuned = trainer.model
tuned.config.use_cache = True
tuned.eval()

tuned.save_pretrained(WORK / "glossary_adapter")

Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

... (2 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,2.864965
20,1.835936
30,1.243135
40,0.623585
50,0.307843
60,0.156473
70,0.131509
80,0.103715
90,0.099159
100,0.082998


Training time: 293.68
Steps: 130
Epochs: 10.0


### 4b. If Training Stops or Cannot Run

**Stopped midway (runtime disconnected, etc.)**: Checkpoints are saved every 10 steps in `/content/qlora_lab/ckpt/`. If the files remain, rerun sections 0–3, then change `trainer.train()` in the training cell above to `trainer.train(resume_from_checkpoint=True)` and run it. Training resumes from the last checkpoint up to 40 steps (20 epochs). If the runtime and its files are gone, restart training or use the alternative below.

**Could not train (no GPU available or not enough time)**: Compare and discuss the instructor's results (outputs and scores after training) from the original Japanese notebook run on a T4. Cells from section 5 onward require the trained model `tuned`, so read the explanations alongside the instructor's results. When submitting, set `executed_training_myself` in section 7 to `False` and state clearly how far you ran the notebook yourself.

## 5. Evaluate After Training (Same test and Generation Settings as Before)

**What to Do**: Evaluate the trained model `tuned` on exactly the same 21 questions and generation settings, and compare before and after. The only change is whether adapters are present.

**What to Look For**

- Comparison table: how did `format_rate` (format compliance rate) and `content_rate` (content accuracy) change from before training?
- `Regressions`: questions that passed both format and content before training but failed afterward. Ideally the list is empty; inspect any regressions in your own run.
- Per-question table: read the remaining failures and identify their types. The original Japanese run included these types:
  - Applying a different FAQ item and making an assertion for a question without evidence (expected `needs_review` is `True`).
  - Evading a question with evidence (expected value is `False`) by requesting confirmation and saying "It is not listed".

In the original Japanese run, after-training answers tended to reproduce the training text (the original FAQ wording) almost verbatim. Check whether your outputs show the same pattern. Format and wording become consistent, but the model has not necessarily learned to judge "whether the FAQ contains evidence".

GPU training is not perfectly reproducible even with a fixed seed. Format compliance and content accuracy may be nearly the same while the remaining questions differ between runs. It is normal for your "remaining questions" to differ from your classmate's; compare failure types.

In [20]:
# Evaluate after LoRA training
after_rows = evaluate_glossary(tuned, "after")

# Compare before and after
comparison = []

for before, after in zip(before_rows, after_rows):
    comparison.append({
        "type": before["type"],
        "question": before["question"],
        "expected": before["expected"],
        "before": before["output"],
        "after": after["output"]
    })

comparison_df = pd.DataFrame(comparison)
display(comparison_df)

# Save comparison
(WORK / "glossary_comparison.json").write_text(
    json.dumps(comparison, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Comparison saved!")
print("Questions evaluated:", len(comparison))

after: 7 questions evaluated


,type,question,expected,output
0,known,Define LoRA.,"A PEFT method that freezes pretrained weights,...","A PEFT method that freezes pretrained weights,..."
1,known,What does AI engineering mean?,"The technical field of designing, building, ev...","The technical field of designing, building, ev..."
2,reworded,How does LoRA reduce the number of trainable p...,"A PEFT method that freezes pretrained weights,...",LoRA replaces the original model's weights wit...
3,reworded,Why do developers use a devcontainer?,A mechanism for defining a development environ...,A mechanism for defining a development environ...
4,reworded,Explain the build-measure-improve cycle.,A development approach that repeats build → me...,A development approach in which build → measur...
5,unknown,What is Speculative decoding?,Not covered in the glossary,A method that predicts the next token using sp...
6,unknown,What is Mixture of Depths?,Not covered in the glossary,A method that combines the results of multiple...


,type,question,expected,before,after
0,known,Define LoRA.,"A PEFT method that freezes pretrained weights,...",**LoRA** stands for **Low-Rank Adaptation**. I...,"A PEFT method that freezes pretrained weights,..."
1,known,What does AI engineering mean?,"The technical field of designing, building, ev...",**AI Engineering** (Artificial Intelligence En...,"The technical field of designing, building, ev..."
2,reworded,How does LoRA reduce the number of trainable p...,"A PEFT method that freezes pretrained weights,...",LoRA (Low-Rank Adaptation) reduces the number ...,LoRA replaces the original model's weights wit...
3,reworded,Why do developers use a devcontainer?,A mechanism for defining a development environ...,Developers use **DevContainers** (often called...,A mechanism for defining a development environ...
4,reworded,Explain the build-measure-improve cycle.,A development approach that repeats build → me...,The **Build-Measure-Improve (BMI)** cycle is a...,A development approach in which build → measur...
5,unknown,What is Speculative decoding?,Not covered in the glossary,**Speculative decoding** is a technique used i...,A method that predicts the next token using sp...
6,unknown,What is Mixture of Depths?,Not covered in the glossary,**Mixture of Depths** (often referred to as **...,A method that combines the results of multiple...


Comparison saved!
Questions evaluated: 7


## 6. Knowledge Questions (Learning a Format Does Not Guarantee Knowledge)

**What to Do**: Separately from the format experiment, ask 3 factual questions absent from both the FAQ and training data (plus one FAQ-covered question as a control). The desired response is "Confirmation is needed (`needs_review: true`)".

Classify responses into three categories.

| Classification | Meaning |
| --- | --- |
| `format_error` | Does not follow the JSON format (common before training in the original Japanese run) |
| `needs_review` | Returns "It is not listed. Confirmation is needed" (the desired response) |
| `asserted` | Makes an assertion with `needs_review: false`. For a question absent from the FAQ, this is an unsupported assertion (fabrication) |

**What to Look For**: In the original Japanese run, format errors were common before training, and both "confirmation needed" and "assertion" responses appeared after training. Check the response types in your own run. The table also shows original outputs. For assertions, read what the model seems to rely on (whether it reuses a different FAQ item). Report results for this dataset rather than deciding in advance that "fine-tuning can never correct knowledge".

Before-training outputs are obtained by temporarily disabling adapters (`tuned.disable_adapter()`).

In [23]:
# Check questions not covered by the glossary

unknown_results = [
    row for row in comparison
    if row["type"] == "unknown"
]

for row in unknown_results:
    print("Question:", row["question"])
    print("Before:", row["before"])
    print("After:", row["after"])
    print("------------------------")

Question: What is Speculative decoding?
Before: **Speculative decoding** is a technique used in **machine learning**, specifically within **generative models** (such as diffusion models, GANs, and VAEs), to improve the quality of generated images or audio.

Instead of generating a single, perfect image or audio track at once, the model generates a sequence of intermediate "samples" (often called "tokens" or "images"). These samples are then processed sequentially through a decoding process to reconstruct the final output
After: A method that predicts the next token using speculative generation and then checks its correctness one step later using evaluation data. This can detect token selection errors early. In Pydantic AI, it is implemented with `GeneratorTaskSpeculativeDecoding`.
------------------------
Question: What is Mixture of Depths?
Before: **Mixture of Depths** (often referred to as **Mixture of Depths** or **Mixture of Depths**) is a concept in **computer graphics**, specifi

## 7. Comparison Table and Submission

**What to Do**: Combine settings, library versions, before/after summaries, and regressed questions into one `result.json`.

**What to Submit**

- `result.json`, `eval_before.json`, and `eval_after.json` (in `/content/qlora_lab`; right-click in the file list on the left → **Download**. With `USE_DRIVE = True`, they are also copied to Drive)
- The comparison table from section 5 and an explanation of remaining failures (failure types)
- Results from section 6 (responses that changed to confirmation requests and those that remained assertions)
- How far you ran training yourself. If you used the instructor's results, set `executed_training_myself` to `False`.

In [24]:
import json

loss_logs = [
    x for x in trainer.state.log_history
    if "loss" in x
]

result = {
    "level": "Advanced",
    "model": CONFIG["base_model"],
    "method": "LoRA",
    "epochs": 10,
    "train_examples": len(train_ds),
    "valid_examples": len(valid_ds),
    "test_questions": len(test_cases),
    "training_time": round(training_seconds, 2),
    "final_loss": loss_logs[-1]["loss"],
    "comparison": comparison
}

output = WORK / "glossary_result.json"

output.write_text(
    json.dumps(result, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("Saved:", output)
print("Training time:", result["training_time"])
print("Final loss:", result["final_loss"])
print("Questions:", result["test_questions"])

Saved: /content/qlora_lab/glossary_result.json
Training time: 293.68
Final loss: 0.07733613848686219
Questions: 7


## 8. Advanced: Load with 4-bit Quantization and Compare Memory and Outputs (Optional)

**What to Do**: Load the same model in 4-bit (NF4) and observe changes in weight memory and before-training outputs. QLoRA adds LoRA to this 4-bit base model and trains it. No training is done here.

- bitsandbytes requires a CUDA GPU, so this runs only in a GPU runtime. Installation takes about 1 minute.
- 4-bit loading places the model on the GPU with `device_map="auto"` (matching bitsandbytes' loading method).
- If you also try training the 4-bit model, run `prepare_model_for_kbit_training(model_4bit)` first, follow section 4's training procedure, and evaluate both before and after using a 4-bit base model.

**What to Look For**

- Weight memory: in the original Japanese run, 4-bit used less than 1GB, versus about 3GB at 32-bit. Compare the displayed values in your run.
- Comparison and per-question tables: since no training occurred, differences from the 32-bit before-training results (`before`) reflect quantization. In the original Japanese run, some questions improved, while other outputs had `needs_review` as the string `"false"` and failed type checks, or asserted "yes" or "none" for questions without evidence. Check the kinds and direction of changes in your run.

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # Storage precision: 4-bit (NF4)
                         bnb_4bit_compute_dtype=torch.float16,             # Compute precision: 16-bit (T4 does not support bf16)
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("Weight memory (GB) | No quantization (32-bit):", BASE_MEMORY_GB, "| 4-bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# No training occurred, so differences from the unquantized before-training results (before) reflect quantization
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"Question": r["question"], "Expected needs_review": r["gold_needs_review"], "4-bit before-training output": r["raw"],
                       "Format/content": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "Format failure reason": r["format_reason"]} for r in q_rows]))